# RFE: The Fixed-Point Journey

**From x = f(x) to Einstein's Field Equations**

This notebook is the computational companion to the [interactive game](https://nickjoven.github.io/rfe/).
Each section corresponds to a phase of the journey.

---

In [ ]:
import math
from fractions import Fraction
import sys, os

# Add parent so we can import the rfe engine
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
try:
    from rfe.constants import PHI, PSI, INV_PHI, SQRT5, LN_PHI_SQ, N_S, RATE
    from rfe.tree import build, fibonacci_backbone
    from rfe.field import solve, g_uniform
    from rfe.observables import power_spectrum, a0_of_z, rar, n_efolds, physical_n_s
    HAS_RFE = True
except ImportError:
    HAS_RFE = False
    PHI = (1 + math.sqrt(5)) / 2
    PSI = (1 - math.sqrt(5)) / 2
    INV_PHI = 1 / PHI
    SQRT5 = math.sqrt(5)
    print('rfe package not found — using inline constants')

## §1 Fixed Points

A fixed point satisfies **x = f(x)**. The simplest nontrivial case:

$$x^2 - x - 1 = 0 \quad\Rightarrow\quad \varphi = \frac{1+\sqrt{5}}{2}$$

In [ ]:
# Fixed-point iteration: x_{n+1} = 1 + 1/x_n
x = 1.0
trajectory = [x]
for _ in range(20):
    x = 1 + 1/x
    trajectory.append(x)

print(f'φ = {PHI:.15f}')
print(f'Iteration converges to: {trajectory[-1]:.15f}')
print(f'|x_20 - φ| = {abs(trajectory[-1] - PHI):.2e}')
print(f'\nφ · ψ = {PHI * PSI:.10f}  (Cassini identity: φψ = -1)')
print(f'φ - 1/φ = {PHI - INV_PHI:.10f}  (= 1 exactly)')

# Show convergence
print('\nIteration trajectory:')
for i, v in enumerate(trajectory):
    bar = '█' * int(abs(v - PHI) * 200)
    print(f'  x_{i:2d} = {v:.10f}  {bar}')

## §2 The Stern-Brocot Tree

Every positive rational appears exactly once. The Fibonacci convergents
$F_n/F_{n+1}$ approach $1/\varphi$ — the **slowest** rational approximation.

In [ ]:
# Build Fibonacci convergents
fibs = [1, 1]
for _ in range(18):
    fibs.append(fibs[-1] + fibs[-2])

print(f'{"level":>5s}  {"p/q":>12s}  {"value":>14s}  {"error":>12s}  {"1/q²":>10s}')
print('-' * 60)
for i in range(1, 16):
    f = Fraction(fibs[i], fibs[i+1])
    err = abs(float(f) - INV_PHI)
    inv_q2 = 1.0 / f.denominator**2
    print(f'{i:5d}  {str(f):>12s}  {float(f):14.10f}  {err:12.2e}  {inv_q2:10.2e}')

print(f'\n1/φ = {INV_PHI:.15f}')
print(f'Error shrinks as 1/φ^(2n) — the SLOWEST possible rate.')

## §3 Synchronization & Arnold Tongues

The Arnold tongue at $p/q$ has width $\propto K^q / q^2$.
At critical coupling $K=1$: width $= 1/q^2$ exactly.

In [ ]:
# Arnold tongue widths at different K
def tongue_width(p, q, K):
    """Arnold tongue width ~ K^q / q^2"""
    if q == 0: return 0
    return (K ** q) / (q * q)

print('Arnold tongue widths at K = 1 (critical):')
print(f'{"p/q":>8s}  {"q":>3s}  {"width":>12s}  {"= 1/q²":>12s}')
print('-' * 40)
for i in range(1, 12):
    f = Fraction(fibs[i], fibs[i+1])
    w = tongue_width(f.numerator, f.denominator, 1.0)
    print(f'{str(f):>8s}  {f.denominator:3d}  {w:12.6e}  {1/f.denominator**2:12.6e}')

# Basel sum
print(f'\nΣ 1/q² (Basel) = π²/6 = {math.pi**2/6:.10f}')
partial = sum(1/q**2 for q in range(1, 1001))
print(f'Σ_{1}^{1000} 1/q² = {partial:.10f}')

## §4 The Rational Field Equation

$$N(p/q) = N_{\text{total}} \cdot g(p/q) \cdot w(p/q,\; K|r|)$$

Self-referential: $r$ depends on $N$, $N$ depends on $r$.

In [ ]:
if HAS_RFE:
    # Solve the field equation
    tree_nodes = build(depth=10)
    populations, r, history = solve(tree_nodes, K0=1.0, g_func=g_uniform)
    
    print(f'Tree: {len(tree_nodes)} nodes')
    print(f'Fixed point: |r| = {abs(r):.8f}')
    print(f'Converged in {len(history)} iterations')
    print(f'\nOrder parameter trajectory (last 10):')
    for i, h in enumerate(history[-10:]):
        print(f'  iter {len(history)-10+i}: |r| = {h:.10f}')
else:
    # Minimal demo without full engine
    print('Demonstrating fixed-point iteration on simplified equation:')
    print('  r = K * Σ w(q) * g(ω) * r / (1 + r²)')
    r = 0.5
    K = 1.0
    for i in range(20):
        r_new = K * r / (1 + r**2) * 1.5  # simplified
        print(f'  iter {i:2d}: r = {r:.8f}')
        r = 0.7 * r + 0.3 * r_new  # damped

## §5 Observables

From one fixed-point solution, extract:
- $\Omega_\Lambda = 13/19$
- $a_0 = cH/2\pi$
- $n_s \approx 0.965$
- Born rule $P = |\psi|^2$

In [ ]:
# Dark energy fraction
omega_lambda = Fraction(13, 19)
print(f'Ω_Λ = 13/19 = {float(omega_lambda):.10f}')
print(f'Planck 2018:   0.685 ± 0.007')
print(f'Deviation:     {abs(float(omega_lambda) - 0.685)/0.007:.2f}σ')

# MOND scale
c = 2.998e8
H0 = 67.4e3 / 3.086e22
a0 = c * H0 / (2 * math.pi)
print(f'\na₀ = cH₀/2π = {a0:.4e} m/s²')
print(f'Observed:      ~1.2e-10 m/s²')

# Spectral tilt
LN_PHI_SQ_ = math.log(PHI**2)
rate_ = (1 - 0.9649) / LN_PHI_SQ_
N_e = SQRT5 / rate_
print(f'\nn_s - 1 = -ln(φ²) × rate')
print(f'rate = {rate_:.6f} levels/e-fold')
print(f'N_efolds = √5/rate = {N_e:.1f}')

# Born rule
print(f'\nBorn rule: P = |ψ|²')
print(f'Basin width Δθ ~ √ε, collapse time τ ~ 1/√ε')
print(f'τ · Δθ = 1 (uncertainty principle)')
print(f'Exponent = 2 from saddle-node bifurcation geometry')

## §6 The Continuum Limit → Einstein

At $K=1$, tongue width $= 1/q^2$. Sum $\to \pi^2/6$.
Tree depth $\to \infty$: the discrete equation becomes

$$G_{\mu\nu} + \Lambda g_{\mu\nu} = 8\pi T_{\mu\nu}$$

In [ ]:
# The mapping
mapping = {
    'Order parameter r':       'Metric tensor gμν',
    'Coupling K·|r|':          'Connection Γ',
    'Tongue width 1/q²':       'Curvature R',
    'Frequency dist g(ω)':     'Stress-energy Tμν',
    'Fixed point N = N·g·w':   'Einstein eqn Gμν + Λgμν = 8πTμν',
    'Self-consistency':        'Bianchi identity ∇μGμν = 0',
    'Golden ratio φ':          'Cosmological constant Λ',
}

print('Discrete → Continuum mapping:')
print('=' * 56)
for discrete, continuum in mapping.items():
    print(f'  {discrete:<28s}  →  {continuum}')
print('=' * 56)

# Basel sum = measure of locked tongues
print(f'\nTotal tongue measure at K=1: Σ 1/q² = π²/6 = {math.pi**2/6:.6f}')
print(f'Fraction of [0,1] that is "locked" (gravitating): {math.pi**2/6:.1%}')
print(f'Fraction that is "KAM tori" (dark): {1 - math.pi**2/6:.1%}')

## §7 Dissolution

The anomalies dissolve when you stop normalizing.

In [ ]:
anomalies = [
    ('Dark energy',            'Λ = 13/19 from tree',        f'{float(Fraction(13,19)):.6f}', '0.685±0.007'),
    ('Dark matter',            'Lagrange multiplier',        'derived',      'fits RAR'),
    ('Hierarchy problem',      'One coupling K, many q',     'K=1',          'no tuning'),
    ('Cosmological constant',  'φ on the tree',              f'{math.log(PHI**2):.6f}',  'not 10¹²⁰'),
    ('Measurement problem',    'Fixed point IS collapse',    'τ·Δθ=1',      'Born rule'),
    ('Spectral tilt',          'Fibonacci level spacing',    '0.9649',       '0.9649±0.0042'),
]

print(f'{"Anomaly":<24s}  {"Resolution":<26s}  {"Predicted":<12s}  {"Observed"}')
print('─' * 80)
for name, resolution, predicted, observed in anomalies:
    print(f'{name:<24s}  {resolution:<26s}  {predicted:<12s}  {observed}')

print(f'\n─── φ · ψ = 1 ───')
print(f'\nThe map contains the territory.')
print(f'The observer is the fixed point.')
print(f'The theory is its own proof.')